# Skin Cancer Classification — EDA & ABCD Feature Analysis
**Dataset:** HAM10000 (Human Against Machine with 10000 training images)  
**Target:** Classify skin lesions into 7 diagnostic categories  
**Features:** ABCD (Asymmetry, Border, Color, Diameter) + patient metadata

---

In [ ]:
import os, sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

from feature_extraction import preprocess, compute_asymmetry, compute_border, compute_color, compute_diameter

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# ── Update these paths ──────────────────────────────────────────
METADATA_CSV = '../data/HAM10000_metadata.csv'
IMAGE_DIR    = '../data/images/'
# ────────────────────────────────────────────────────────────────

LABEL_MAP = {
    'mel':   'Melanoma',
    'nv':    'Melanocytic nevi',
    'bcc':   'Basal cell carcinoma',
    'akiec': 'Actinic keratosis',
    'bkl':   'Benign keratosis',
    'df':    'Dermatofibroma',
    'vasc':  'Vascular lesion',
}
MALIGNANT = {'mel', 'bcc', 'akiec'}
COLORS = ['#534AB7', '#1D9E75', '#D85A30', '#D4537E', '#639922', '#BA7517', '#378ADD']

df = pd.read_csv(METADATA_CSV)
df['label'] = df['dx'].map(LABEL_MAP)
df['is_malignant'] = df['dx'].isin(MALIGNANT)
print(f'Dataset: {len(df)} rows, {df["dx"].nunique()} classes')
df.head()

## 1. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count bar
counts = df['label'].value_counts()
bars = axes[0].barh(counts.index, counts.values, color=COLORS)
axes[0].set_xlabel('Number of images')
axes[0].set_title('Class distribution', fontweight='bold')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=10)

# Malignant vs Benign
mal_counts = df['is_malignant'].value_counts()
labels = ['Benign', 'Malignant']
axes[1].pie([mal_counts[False], mal_counts[True]], labels=labels,
            colors=['#1D9E75', '#D85A30'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Malignant vs Benign split', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Patient Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age distribution by class
for i, (cls, grp) in enumerate(df.groupby('dx')):
    axes[0].hist(grp['age'].dropna(), alpha=0.5, bins=20, label=LABEL_MAP[cls], color=COLORS[i])
axes[0].set_xlabel('Age')
axes[0].set_title('Age distribution by class', fontweight='bold')
axes[0].legend(fontsize=8)

# Sex distribution
sex_dx = df.groupby(['dx', 'sex']).size().unstack(fill_value=0)
sex_dx.index = [LABEL_MAP[c] for c in sex_dx.index]
sex_dx[['male', 'female']].plot(kind='barh', ax=axes[1], color=['#534AB7', '#D4537E'])
axes[1].set_title('Sex distribution by class', fontweight='bold')
axes[1].set_xlabel('Count')

# Localization
loc_counts = df['localization'].value_counts().head(10)
axes[2].barh(loc_counts.index, loc_counts.values, color='#534AB7', alpha=0.8)
axes[2].set_title('Top 10 lesion locations', fontweight='bold')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('../outputs/eda_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. ABCD Feature Extraction — Visual Demo

In [ ]:
# Pick one sample image from each class for visualization
sample_images = []
for cls in LABEL_MAP.keys():
    row = df[df['dx'] == cls].iloc[0]
    path = os.path.join(IMAGE_DIR, f"{row['image_id']}.jpg")
    if os.path.exists(path):
        sample_images.append((cls, path))

fig, axes = plt.subplots(len(sample_images), 4, figsize=(16, len(sample_images)*3))
fig.suptitle('ABCD Feature Extraction per Class', fontsize=16, fontweight='bold', y=1.01)

for row_idx, (cls, path) in enumerate(sample_images):
    image, mask = preprocess(path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # A: Original + mask overlay
    overlay = image_rgb.copy()
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (255, 80, 80), 2)
    axes[row_idx, 0].imshow(overlay)
    asym = compute_asymmetry(mask)
    axes[row_idx, 0].set_title(f'A: Asymmetry={asym["asymmetry_score"]:.3f}\n{LABEL_MAP[cls]}', fontsize=9)

    # B: Contour on white
    border_img = np.ones((*image_rgb.shape[:2], 3), dtype=np.uint8) * 245
    cv2.fillPoly(border_img, contours, (200, 200, 240))
    cv2.drawContours(border_img, contours, -1, (83, 74, 183), 2)
    bord = compute_border(mask)
    axes[row_idx, 1].imshow(border_img)
    axes[row_idx, 1].set_title(f'B: Irregularity={bord["border_irregularity"]:.3f}', fontsize=9)

    # C: HSV heatmap of H channel
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hue = hsv[:, :, 0].astype(float)
    hue[mask == 0] = np.nan
    col = compute_color(image, mask)
    axes[row_idx, 2].imshow(hue, cmap='hsv', vmin=0, vmax=180)
    axes[row_idx, 2].set_title(f'C: Entropy={col["color_entropy"]:.3f}', fontsize=9)

    # D: Bounding box
    bbox_img = image_rgb.copy()
    if contours:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        cv2.rectangle(bbox_img, (x, y), (x+w, y+h), (29, 158, 117), 2)
    diam = compute_diameter(mask)
    axes[row_idx, 3].imshow(bbox_img)
    axes[row_idx, 3].set_title(f'D: MaxDiam={diam["diameter_max_mm"]:.2f}mm', fontsize=9)

    for ax in axes[row_idx]:
        ax.axis('off')

plt.tight_layout()
plt.savefig('../outputs/eda_abcd_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Load Pre-Extracted ABCD Features and Train Model

In [ ]:
# Run this after running train_model.py with --cache_features
FEATURE_CACHE = '../data/abcd_features_cache.csv'

if os.path.exists(FEATURE_CACHE):
    abcd_df = pd.read_csv(FEATURE_CACHE)
    abcd_df['image_id'] = abcd_df['image_path'].apply(
        lambda p: os.path.splitext(os.path.basename(p))[0]
    )
    merged = df.merge(abcd_df, on='image_id', how='inner')
    print(f'Merged: {len(merged)} rows')

    # Correlation heatmap of ABCD features
    abcd_cols = [c for c in abcd_df.columns if c not in ['image_path', 'image_id', 'error']]
    corr = merged[abcd_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, ax=ax, annot_kws={'size': 8})
    ax.set_title('ABCD Feature Correlation Matrix', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig('../outputs/eda_feature_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Box plots: key ABCD features by class
    key_features = ['asymmetry_score', 'border_irregularity', 'color_entropy', 'diameter_max_mm']
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    for ax, feat in zip(axes, key_features):
        if feat in merged.columns:
            order = merged.groupby('dx')[feat].median().sort_values(ascending=False).index
            sns.boxplot(data=merged, x='dx', y=feat, order=order, ax=ax,
                        palette=COLORS[:len(order)], linewidth=0.8)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
            ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    plt.suptitle('ABCD Feature Distribution by Diagnosis Class', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/eda_abcd_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Feature cache not found at {FEATURE_CACHE}.')
    print('Run: python ../src/train_model.py --cache_features ../data/abcd_features_cache.csv ...')

## 5. Quick Single-Image ABCD Inspection

In [ ]:
from feature_extraction import extract_all_features

# Change this to any image path
TEST_IMAGE = os.path.join(IMAGE_DIR, df.iloc[0]['image_id'] + '.jpg')

if os.path.exists(TEST_IMAGE):
    feats = extract_all_features(TEST_IMAGE)
    print('ABCD Feature Report')
    print('=' * 40)
    for k, v in feats.items():
        if k != 'image_path':
            print(f'  {k:<35} {v}')
else:
    print(f'Image not found: {TEST_IMAGE}')